# Week 8 Demo: Regression Workflows

## What are we trying to predict?
An IT team wants to estimate how many minutes a service request will take. This could help the team plan its workload.

We will use information collected when the request arrives to predict the work time. Each row describes one completed request. The data was created for practice; it does not describe real customers or jobs.

We will use `LinearRegression()` with the data-preparation steps listed in `DATA_SOURCE.md`. That file explains what each column means and where the data comes from.

## What counts as a useful result in this example?
We set these practice goals before training the model:

- The model's average absolute error on the test requests must be **12 minutes or less**.
- That error must be lower than the error from a simple guess: use the middle work time from the training data for every request.

We will call that simple guess our **baseline**. It gives us something to compare the model with.

Open and run the notebook from the demo folder using the course Python environment. We will keep work time in minutes throughout the example.


## Step 1: Import the tools

We will reuse the tools for filling missing values, encoding categories, and scaling numbers. We will also import tools to split the data, train a linear regression model, and measure prediction error.

The first output shows the scikit-learn version used by your selected Python environment.


In [ ]:
from pathlib import Path
import sklearn
print("scikit-learn version:", sklearn.__version__)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


## Step 2: Inspect the data

Each row describes one completed service request. Start by looking at the first few rows, the number of rows and columns, and the missing values.

We want to predict `actual_work_minutes`. This is our **target**: the value we want the model to estimate.

Our input columns, or **features**, must contain information we know when a request arrives. `resolution_steps` is recorded after the work is done, so we cannot use it to make that earlier estimate.


In [ ]:
df = pd.read_csv("data/week8_service_requests.csv")
display(df.head())
print("Shape:", df.shape)
display(df.isna().sum())


## Step 3: Define the target and features

Put the input columns in `X` and the actual work times in `y`.

- Remove the rows with missing work times. We need known actual values to train the model and check its predictions.
- Keep rows with missing input values. The pipeline will fill those values using the rules we have chosen.
- Leave out `request_id`. It identifies a request but is not one of our chosen inputs.
- Leave out `resolution_steps`, because we do not know the value of that field when the request arrives.

The assertions check that these basic requirements are met.


In [ ]:
target = "actual_work_minutes"
numeric_features = ["device_count", "complexity_score"]
categorical_features = ["request_channel", "system_type"]
feature_columns = numeric_features + categorical_features
# Remove only rows without a known target. Leave missing features for the pipeline.
working_df = df.dropna(subset=[target]).copy()
X = working_df[feature_columns].copy()
y = working_df[target].copy()
assert y.notna().all()
assert (y > 0).all()
assert df["request_id"].is_unique
assert target not in X.columns
assert "request_id" not in X.columns
assert "resolution_steps" not in X.columns
print("Rows retained:", len(working_df))


## Step 4: Split before fitting

Divide the rows into two groups:

- **Training data:** 75% of the rows. Use these rows to learn the data-preparation values and train the model.
- **Test data:** 25% of the rows. Save these rows to check the model's predictions after training.

Split first so the test rows do not influence the missing-value replacements, scaling values, or model training.

`random_state=8` gives us the same split each time we run this example. A random split fits this practice dataset because each row describes a separate request. It does not guarantee that the test rows represent every request we might receive in the future.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=8
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))
display(X_train.describe())


## Step 5: Build the data-preparation steps and model

Use a different set of preparation steps for each feature group:

- **Numerical features:** fill missing values with the training median, then apply `StandardScaler`.
- **Categorical features:** fill missing values with the most common training category, then apply `OneHotEncoder`.

`ColumnTransformer` applies the right steps to each group. `Pipeline` connects those preparation steps to `LinearRegression`.

We have only defined the steps here. The next step will use the training data to learn the values they need.

We continue scaling to practice the workflow from earlier weeks. Linear regression generally does not need scaling to make predictions. We scale the input features, while leaving the target in minutes.

### A note about category coefficients
A coefficient is a number the model learns for an input column when it calculates predictions.
With full one-hot encoding, some columns repeat information already available in the other columns. The model can still make predictions, but we should not read each category's coefficient as a separate, clear effect.

If we later study category coefficients, `drop="first"` can leave out one category so the others can be compared with it. Keep the encoder as written for this demo; our focus is on predictions.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")
regression_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])


## Step 6: Train the model and make test predictions

`fit(X_train, y_train)` learns the missing-value replacements, scaling values, and categories from the training rows. It then trains the linear regression model using those prepared inputs and their actual work times.

`predict(X_test)` uses those same preparation values to process the test inputs and estimate their work times.

The results table pairs each estimate with the correct actual work time. The absolute error is the size of the difference between them, regardless of whether the estimate is too high or too low.


In [ ]:
regression_pipeline.fit(X_train, y_train)
predictions = regression_pipeline.predict(X_test)
results = pd.DataFrame({
    "actual_minutes": y_test,
    "predicted_minutes": predictions,
})
results["absolute_error_minutes"] = (
    results["actual_minutes"] - results["predicted_minutes"]
).abs()
display(results.head())


## Step 7: Measure error and compare with a simple guess

### MAE: How far off are the predictions?
**Mean absolute error (MAE)** is the average size of the prediction errors. It is measured in minutes here. Lower is better.

For example, errors of 5, 10, and 15 minutes have an MAE of 10 minutes. Some requests can have errors larger than that average.

### Baseline: Does the model improve on a simple guess?
The baseline finds the median, or middle work time, in the training data. It predicts that same number for every test request.

Compare the model and baseline using the same test requests. Our model must have MAE of **12 minutes or less** and MAE **lower than the baseline's** to meet both practice goals.

### R²: A second way to describe performance
R² compares the model with always predicting the average actual work time in the test data. This comparison uses squared errors, which give larger mistakes more weight.

- **1:** every test prediction is correct.
- **0:** the model has the same total squared error as that average-value guess.
- **Below 0:** the model has more total squared error than that guess.

R² has no units. It is not a percentage of correct predictions. Its average-value comparison is different from our training-median baseline.

We show R² as an introduction. Our two practice goals still use MAE. Use the original scores to check the goals, and round only when displaying results.


In [ ]:
model_mae = mean_absolute_error(y_test, predictions)
# Introductory companion score: higher is better; this has no minute units.
model_r2 = r2_score(y_test, predictions)
# The baseline ignores features and always predicts the training-target median.
baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
MAE_LIMIT = 12.0
meets_error_limit = bool(model_mae <= MAE_LIMIT)
beats_baseline = bool(model_mae < baseline_mae)
meets_criteria = meets_error_limit and beats_baseline
metrics = pd.DataFrame({
    "workflow": ["Linear regression", "Training-median baseline"],
    "test_mae_minutes": [model_mae, baseline_mae],
})
display(metrics)
print(f"Model test R² (introductory): {model_r2:.3f}")
print(f"Error limit: MAE <= {MAE_LIMIT:.1f} minutes: {meets_error_limit}")
print("Model MAE lower than baseline MAE:", beats_baseline)
print("Both instructional criteria met:", meets_criteria)


## Step 8: Check the workflow

Use assertions to check that:

- training and test data contain different rows;
- each input row is paired with the correct target value;
- preparation keeps the expected number of rows;
- prepared training and test inputs have the same number of feature columns;
- prepared values contain no missing values or infinity;
- there is one prediction for each test request;
- averaging the errors in our results table gives the same MAE as the metric function.

We also check that the imputer's stored medians match the medians from the training data.

Use `transform` here to reuse the preparation steps that have already learned from training data. Do not fit them again on the test rows.


In [ ]:
# Inspect the already-fitted preprocessing. Calling transform reuses training statistics.
fitted_preprocessor = regression_pipeline.named_steps["preprocessor"]
train_transformed = fitted_preprocessor.transform(X_train)
test_transformed = fitted_preprocessor.transform(X_test)
feature_names = fitted_preprocessor.get_feature_names_out()
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert len(X_train) + len(X_test) == len(working_df)
assert train_transformed.shape[0] == len(X_train)
assert test_transformed.shape[0] == len(X_test)
assert train_transformed.shape[1] == test_transformed.shape[1] == len(feature_names)
assert np.isfinite(train_transformed).all()
assert np.isfinite(test_transformed).all()
assert predictions.shape == (len(y_test),)
assert np.isfinite(predictions).all()
# This specifically checks where the imputer learned its medians.
fitted_imputer = fitted_preprocessor.named_transformers_["numeric"].named_steps["imputer"]
assert np.allclose(fitted_imputer.statistics_, X_train[numeric_features].median().to_numpy())
assert np.isclose(model_mae, results["absolute_error_minutes"].mean())
print("Verification checks passed.")
print("Transformed training/test shapes:", train_transformed.shape, test_transformed.shape)
print("Feature names:", list(feature_names))


## Step 9: Predict work time for new requests

These new requests have no actual work times yet. We can predict their work times, but we cannot check their errors until the work is completed.

The second request has a missing device count. The existing pipeline fills it using the median it learned from the training data.

Use `predict` with the existing pipeline. We do not train it again on these new requests.


In [ ]:
new_requests = pd.DataFrame({
    "device_count": [6.0, np.nan],
    "complexity_score": [3.0, 2.0],
    "request_channel": ["portal", "phone"],
    "system_type": ["network", "desktop"],
})
new_predictions = regression_pipeline.predict(new_requests)
assert new_predictions.shape == (len(new_requests),)
assert np.isfinite(new_predictions).all()
new_results = new_requests.copy()
new_results["predicted_work_minutes"] = new_predictions
display(new_results)


## Step 10: Explain the results and their limits

The plot compares actual and predicted work times for the test requests. A point on the dashed line is an exact prediction. Points farther from the line have larger errors.

An average error does not tell us the largest error. Passing our practice goals also does not prove the model is ready for a real service team: we used practice data and one training/test split.

Describe what the results show for this example and what we would still need to check before using the model with real requests.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(results["actual_minutes"], results["predicted_minutes"], alpha=0.7)
low = min(results["actual_minutes"].min(), results["predicted_minutes"].min())
high = max(results["actual_minutes"].max(), results["predicted_minutes"].max())
ax.plot([low, high], [low, high], linestyle="--", color="black", label="Perfect prediction")
ax.set(xlabel="Actual work time (minutes)", ylabel="Predicted work time (minutes)",
       title="Held-out test predictions")
ax.legend()
fig.tight_layout()
plt.show()
print(f"Model test MAE: {model_mae:.2f} minutes; baseline: {baseline_mae:.2f} minutes.")
print("Both instructional criteria met:", meets_criteria)


## What would you tell the IT team?

Use the notebook outputs to answer these questions:

1. What is the model's test MAE, in minutes? What does that mean?
2. What is the baseline's test MAE? Does the model have a smaller average error?
3. Does the model meet the 12-minute limit? Does it meet both practice goals?
4. Why did we leave `resolution_steps` out of the inputs?

Let’s use our outputs to answer the four questions.

First, the model’s test MAE is about 8.40 minutes. That means its predictions differ from the actual work times by about 8.40 minutes on average. Some individual errors are larger and some are smaller.

Second, the baseline’s test MAE is about 27.01 minutes. The baseline predicts the median training work time for every test request. Our regression model has a smaller average error than that simple guess.

Third, the model meets the assumed 12-minute limit because 8.40 is below 12. It also beats the baseline, so it meets both practice goals for these test requests. That does not mean every prediction is within 12 minutes of the actual time.

Finally, we left resolution_steps out because those steps are recorded after the work finishes. We want to estimate work time when a request first arrives, so that information would not be available when we need the prediction.

## Sources
- Data School, Chapter 2, Section 2.4: https://mlbook.dataschool.io/ch02.html#qa-how-do-i-adapt-this-workflow-to-a-regression-problem
- Data School, Chapter 4, Sections 4.1–4.3: https://mlbook.dataschool.io/ch04.html
- scikit-learn 1.9, R²: https://scikit-learn.org/1.9/modules/generated/sklearn.metrics.r2_score.html
- scikit-learn 1.9, MAE: https://scikit-learn.org/1.9/modules/generated/sklearn.metrics.mean_absolute_error.html
- scikit-learn 1.9, DummyRegressor: https://scikit-learn.org/1.9/modules/generated/sklearn.dummy.DummyRegressor.html
- scikit-learn 1.9, common pitfalls: https://scikit-learn.org/1.9/common_pitfalls.html
